# Xception — Show regions influencing a prediction

Use case: inspect which image regions contribute to a CNN classification score. Grad-CAM combines gradients with convolution activations to create a heatmap. Xception uses depthwise separable convolutions.

Run the cells from top to bottom. Each notebook is independent.


## Step 1 — Install packages
Use Python 3.11 or 3.12 as a straightforward training environment, or Google Colab. Install into the active notebook kernel. Restart the kernel after the first installation if required. Internet is needed initially for packages, sample images/data and ImageNet weights. A GPU is optional; CPU runs can be slower. Model downloads can be hundreds of MB.


In [ ]:
%pip install tensorflow numpy matplotlib scikit-learn pillow


## Step 2 — Import libraries
TensorFlow runs the neural network. Keras provides the model API. NumPy handles arrays and Matplotlib displays pictures.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)


## Step 3 — Load an image
The heatmap will be placed over this picture.


In [ ]:
image_path = keras.utils.get_file(
    "sunflower.jpg",
    "https://storage.googleapis.com/download.tensorflow.org/example_images/592px-Red_sunflower.jpg"
)
# To use your own picture, replace image_path with a local filename.
original = keras.utils.load_img(image_path)
plt.figure(figsize=(6, 6))
plt.imshow(original)
plt.axis("off")
plt.show()


## Step 4 — Load Xception and preprocess
classifier_activation=None gives logits for gradient calculation. block14_sepconv2_act is the last convolution block activation in this model.


In [ ]:
from tensorflow.keras.applications.xception import Xception, preprocess_input, decode_predictions
model = Xception(weights="imagenet", classifier_activation=None)
picture = keras.utils.load_img(image_path, target_size=(299, 299))
pixels = keras.utils.img_to_array(picture)
batch = preprocess_input(np.expand_dims(pixels, axis=0))
logits = model.predict(batch, verbose=0)
probabilities = tf.nn.softmax(logits, axis=-1).numpy()
class_index = int(np.argmax(probabilities[0]))
label = decode_predictions(probabilities, top=1)[0][0][1]
print("Predicted class:", label)


## Step 5 — Compute a Grad-CAM heatmap
Gradients indicate how the selected class score changes with each feature map. Average the gradients spatially, use them as channel weights, and retain positive contributions.


In [ ]:
gradient_model = keras.Model(model.inputs,
    [model.get_layer("block14_sepconv2_act").output, model.output])
with tf.GradientTape() as tape:
    activations, class_scores = gradient_model(tf.convert_to_tensor(batch), training=False)
    selected_score = class_scores[:, class_index]
gradients = tape.gradient(selected_score, activations)
channel_weights = tf.reduce_mean(gradients, axis=(0, 1, 2))
heatmap = tf.reduce_sum(activations[0] * channel_weights, axis=-1)
heatmap = tf.maximum(heatmap, 0)
heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
heatmap = tf.image.resize(heatmap[..., None], (299, 299)).numpy()[:, :, 0]


## Step 6 — Display original, heatmap and overlay
Warm colors indicate stronger positive contributions to the chosen score. The heatmap is coarse because convolution maps have lower spatial resolution.


In [ ]:
colored = plt.get_cmap("jet")(heatmap)[:, :, :3]
overlay = np.clip(0.65 * pixels / 255.0 + 0.35 * colored, 0, 1)
plt.figure(figsize=(12, 4))
for position, (image_data, title) in enumerate(
    [(pixels.astype("uint8"), "Original"), (heatmap, "Grad-CAM"), (overlay, label)], start=1):
    plt.subplot(1, 3, position)
    plt.imshow(image_data, cmap="jet" if position == 2 else None)
    plt.title(title)
    plt.axis("off")
plt.tight_layout()
plt.show()


## Understand the result
Grad-CAM is an explanation aid, not proof of the model’s reasoning and not object segmentation. A plausible-looking heatmap can accompany an incorrect prediction. To inspect another class, change class_index before computing the heatmap.


## Practice
1. Change the input image or selected image index.
2. Explain the output in your own words.
3. Compare the result with the original image and record one limitation.


## References
- [Keras Applications](https://keras.io/api/applications/)
- [CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html)
- [Keras Grad-CAM example](https://keras.io/examples/vision/grad_cam/)
